# Customer Churn Prediction & Business Insight System

**Zeravia Training & Internship Program — Mini Project Assignment**

**Student Name:** [Enter Your Name]

### Project Objective
The objective is to analyse customer behaviour and build a beginner-friendly machine-learning system that predicts whether a customer is likely to churn. The workflow covers data loading, cleaning, exploratory data analysis, visualisation, statistics, preprocessing, Logistic Regression, evaluation, Linear Regression, feature interpretation, customer risk grouping and business recommendations.

### Dataset
This notebook uses the public **IBM Telco Customer Churn** sample dataset. It contains customer demographic information, subscribed services, account/billing information and a `Churn` target indicating whether the customer left.

**Dataset source:** IBM Telco Customer Churn sample dataset  
**Source URL:** https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv

> If the automatic download does not work in your Jupyter/Colab environment, download the CSV manually and upload it when prompted in the next cell.


## 1. Business Problem

Customer churn means a customer stops using the company's service. Churn is important because losing existing customers can reduce recurring revenue and increase the need to acquire replacement customers.

### Client objective
The client wants to:
- understand which customer characteristics are associated with churn,
- identify customers with higher predicted churn risk,
- build a basic classification model,
- and use the findings to design customer-retention actions.

### Target variable
The target variable is `Churn`:
- `Yes` = customer churned
- `No` = customer did not churn

### Business impact
A churn prediction system can help the client prioritise retention efforts toward customers who appear more likely to leave.


## 2. Python Environment & Project Setup

The project uses:
- Python
- NumPy
- pandas
- Matplotlib
- Seaborn
- scikit-learn

The following cell imports the required libraries.


In [ ]:
import os
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

print("Libraries imported successfully.")


## 3. Data Loading

The dataset is downloaded from the public IBM sample-data repository when possible.

If the download fails, the cell asks for a local CSV file. In Google Colab, upload `WA_Fn-UseC_-Telco-Customer-Churn.csv` when prompted.


In [ ]:
DATA_URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
DATA_FILE = "WA_Fn-UseC_-Telco-Customer-Churn.csv"

try:
    if not Path(DATA_FILE).exists():
        urllib.request.urlretrieve(DATA_URL, DATA_FILE)
    print(f"Dataset ready: {DATA_FILE}")
except Exception as error:
    print("Automatic download failed:", error)
    print("Please download the IBM Telco Customer Churn CSV and place it in the same folder as this notebook.")
    try:
        from google.colab import files
        uploaded = files.upload()
        DATA_FILE = next(iter(uploaded))
        print(f"Uploaded file: {DATA_FILE}")
    except Exception:
        raise FileNotFoundError(
            "Could not download the dataset. Please place the CSV beside this notebook and run this cell again."
        )

df = pd.read_csv(DATA_FILE)
df.head()


## 4. Data Exploration

First, inspect the shape, columns, data types, sample records and basic statistics.


In [ ]:
print("Rows and columns:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst five rows:")
display(df.head())

print("\nData types:")
display(df.dtypes)

print("\nBasic statistics:")
display(df.describe(include="all").T)


In [ ]:
print("Dataset information:")
df.info()

print("\nMissing values by column:")
display(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nUnique values in Churn:")
print(df["Churn"].value_counts())


### Important Feature Meanings

- **tenure**: number of months the customer has stayed with the company.
- **MonthlyCharges**: customer's monthly charge.
- **TotalCharges**: total amount charged to the customer.
- **Contract**: customer's contract type.
- **InternetService**: type of internet service.
- **PaymentMethod**: method used to pay.
- **Churn**: whether the customer left the service.


## 5. Data Cleaning & Preparation

The `TotalCharges` column can contain blank values and may initially be stored as text. It is converted to numeric values using `errors="coerce"`, which turns invalid/blank values into missing values. Those missing values are then replaced with the median.

Customer IDs are identifiers rather than useful predictive measurements, so the ID will be excluded from the machine-learning features later.


In [ ]:
# Convert TotalCharges to numeric
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

print("Missing TotalCharges after numeric conversion:",
      df["TotalCharges"].isnull().sum())

# Fill missing TotalCharges using the median
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())

print("Missing values after cleaning:")
display(df.isnull().sum())

print("Duplicate rows:", df.duplicated().sum())


## 6. Exploratory Data Analysis

The analysis focuses on how churn differs across important customer characteristics such as contract type, monthly charges, tenure and internet service.


In [ ]:
print("Overall churn counts:")
display(df["Churn"].value_counts())

print("\nOverall churn percentage:")
display((df["Churn"].value_counts(normalize=True) * 100).round(2))

print("\nAverage numeric values by churn status:")
display(
    df.groupby("Churn")[["tenure", "MonthlyCharges", "TotalCharges"]]
      .mean()
      .round(2)
)


In [ ]:
print("Churn rate by contract type:")
contract_churn = (
    pd.crosstab(df["Contract"], df["Churn"], normalize="index") * 100
)
display(contract_churn.round(2))

print("Churn rate by internet service:")
internet_churn = (
    pd.crosstab(df["InternetService"], df["Churn"], normalize="index") * 100
)
display(internet_churn.round(2))


## 7. Data Visualisation

At least five meaningful visualisations are required. Each chart below includes a short business interpretation.


### Visualisation 1 — Overall Churn Distribution

In [ ]:
plt.figure(figsize=(7, 5))
sns.countplot(data=df, x="Churn")
plt.title("Customer Churn Distribution")
plt.xlabel("Churn Status")
plt.ylabel("Number of Customers")
plt.show()


**Interpretation:** The chart shows how many customers stayed with the company compared with customers who churned. This also shows whether the target classes are balanced or imbalanced.

### Visualisation 2 — Churn by Contract Type

In [ ]:
plt.figure(figsize=(9, 5))
sns.countplot(data=df, x="Contract", hue="Churn")
plt.title("Customer Churn by Contract Type")
plt.xlabel("Contract Type")
plt.ylabel("Number of Customers")
plt.xticks(rotation=15)
plt.show()


**Interpretation:** Contract type is an important customer-segmentation variable. A visibly larger churn share among shorter contracts would suggest that contract commitment is associated with retention.

### Visualisation 3 — Monthly Charges by Churn

In [ ]:
plt.figure(figsize=(7, 5))
sns.boxplot(data=df, x="Churn", y="MonthlyCharges")
plt.title("Monthly Charges by Churn Status")
plt.xlabel("Churn Status")
plt.ylabel("Monthly Charges")
plt.show()


**Interpretation:** The box plot compares the distribution of monthly charges between customers who churned and those who remained. Differences in the distributions can help identify pricing-related segments that deserve further investigation.

### Visualisation 4 — Tenure Distribution

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(
    data=df,
    x="tenure",
    hue="Churn",
    bins=30,
    kde=True,
    element="step"
)
plt.title("Customer Tenure Distribution by Churn Status")
plt.xlabel("Tenure (months)")
plt.ylabel("Number of Customers")
plt.show()


**Interpretation:** Customers with shorter tenure can be compared with longer-tenure customers to identify whether churn is concentrated earlier in the customer relationship.

### Visualisation 5 — Churn by Internet Service

In [ ]:
plt.figure(figsize=(9, 5))
sns.countplot(data=df, x="InternetService", hue="Churn")
plt.title("Customer Churn by Internet Service Type")
plt.xlabel("Internet Service")
plt.ylabel("Number of Customers")
plt.show()


**Interpretation:** Churn differences between internet-service groups can highlight customer segments that may need service-quality, pricing or support interventions.

### Optional Visualisation 6 — Tenure vs Monthly Charges

In [ ]:
plt.figure(figsize=(9, 5))
sns.scatterplot(
    data=df,
    x="tenure",
    y="MonthlyCharges",
    hue="Churn",
    alpha=0.6
)
plt.title("Tenure vs Monthly Charges by Churn")
plt.xlabel("Tenure (months)")
plt.ylabel("Monthly Charges")
plt.show()


**Interpretation:** The scatter plot shows whether churned and retained customers occupy different regions of the tenure–monthly-charge space.

## 8. Statistical Analysis

Central tendency and dispersion are examined using mean, median, standard deviation, minimum, maximum and quartiles. IQR is used to identify potential outliers.

An outlier is not automatically an error. In a business dataset, it may represent a genuine customer with unusually high charges or a very long tenure.


In [ ]:
numeric_columns = ["tenure", "MonthlyCharges", "TotalCharges"]

statistics_summary = df[numeric_columns].agg(
    ["mean", "median", "std", "min", "max"]
).T

display(statistics_summary.round(2))


In [ ]:
for column in numeric_columns:
    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outliers = df[
        (df[column] < lower_bound) |
        (df[column] > upper_bound)
    ]

    print(f"{column}")
    print(f"  Q1: {q1:.2f}")
    print(f"  Q3: {q3:.2f}")
    print(f"  IQR: {iqr:.2f}")
    print(f"  Lower bound: {lower_bound:.2f}")
    print(f"  Upper bound: {upper_bound:.2f}")
    print(f"  Potential outliers: {len(outliers)}")
    print()


## 9. Data Preprocessing

The target is converted from `Yes/No` to `1/0`.

Categorical input variables are one-hot encoded. `customerID` is removed because it is an identifier and does not represent a meaningful customer characteristic.

Standardisation is applied after the train-test split so that the scaler is fitted only on the training data. This helps prevent information from the test set from influencing preprocessing.


In [ ]:
# Create target
y = df["Churn"].map({"No": 0, "Yes": 1})

# Create feature matrix
X = df.drop(columns=["Churn", "customerID"])

# Convert categorical variables into numeric dummy variables
X = pd.get_dummies(X, drop_first=True)

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

display(X.head())


## 10. Train-Test Split

The dataset is divided into training and testing sets before model evaluation.

- **80% training data** is used to learn model parameters.
- **20% testing data** is kept separate for evaluating performance on unseen records.
- `stratify=y` keeps the churn/non-churn class proportions approximately consistent between the two sets.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training records:", X_train.shape[0])
print("Testing records:", X_test.shape[0])

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Preprocessing completed.")


## 11. Logistic Regression Model

Logistic Regression is used because the target variable is binary: churn or no churn.

The model learns the relationship between the customer features and the probability of churn.


In [ ]:
logistic_model = LogisticRegression(max_iter=1000)

logistic_model.fit(X_train_scaled, y_train)

y_pred = logistic_model.predict(X_test_scaled)
y_probability = logistic_model.predict_proba(X_test_scaled)[:, 1]

print("Logistic Regression model trained successfully.")


## 12. Model Evaluation

The model is evaluated using:
- Accuracy
- Precision
- Recall
- F1-score
- Confusion matrix

From a client perspective, recall is particularly useful when the business wants to identify as many potential churners as possible.


In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

evaluation_results = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score"],
    "Score": [accuracy, precision, recall, f1]
})

display(evaluation_results.round(4))


In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Predicted No Churn", "Predicted Churn"],
    yticklabels=["Actual No Churn", "Actual Churn"]
)
plt.title("Confusion Matrix — Logistic Regression")
plt.xlabel("Prediction")
plt.ylabel("Actual")
plt.show()

print("Confusion matrix:")
print(cm)


### Client Interpretation

**Accuracy** measures the overall proportion of correct predictions.

**Precision** answers: among customers predicted to churn, how many actually churned?

**Recall** answers: among customers who actually churned, how many did the model identify?

**F1-score** balances precision and recall.

For a retention team, recall can be especially important because missing a genuine high-risk customer may mean losing an opportunity to intervene.


## 13. Linear Regression Extension

To demonstrate the difference between classification and regression, a simple Linear Regression model is used to predict `MonthlyCharges` from `tenure`.

- Logistic Regression predicts a class/probability.
- Linear Regression predicts a continuous numerical value.


In [ ]:
X_reg = df[["tenure"]]
y_reg = df["MonthlyCharges"]

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg,
    y_reg,
    test_size=0.20,
    random_state=42
)

linear_model = LinearRegression()
linear_model.fit(X_train_reg, y_train_reg)

y_pred_reg = linear_model.predict(X_test_reg)

mae = mean_absolute_error(y_test_reg, y_pred_reg)
mse = mean_squared_error(y_test_reg, y_pred_reg)
rmse = np.sqrt(mse)
r2 = r2_score(y_test_reg, y_pred_reg)

regression_results = pd.DataFrame({
    "Metric": ["MAE", "MSE", "RMSE", "R²"],
    "Score": [mae, mse, rmse, r2]
})

display(regression_results.round(4))


In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(X_test_reg["tenure"], y_test_reg, alpha=0.5, label="Actual")
plt.scatter(X_test_reg["tenure"], y_pred_reg, alpha=0.5, label="Predicted")
plt.xlabel("Tenure (months)")
plt.ylabel("Monthly Charges")
plt.title("Linear Regression: Predicting Monthly Charges from Tenure")
plt.legend()
plt.show()


### Regression Metric Interpretation

- **MAE** is the average absolute prediction error.
- **MSE** is the average squared prediction error.
- **RMSE** is the square root of MSE and is expressed in the same units as monthly charges.
- **R²** indicates how much variation in the target is explained by the model.

This simple regression is included to demonstrate the difference between a continuous prediction problem and the binary churn classification problem.


## 14. Feature Interpretation

Logistic Regression coefficients indicate the direction and relative strength of the model's learned relationships.

A positive coefficient is associated with higher predicted churn probability, while a negative coefficient is associated with lower predicted churn probability.

These relationships are **associations, not proof of causation**.


In [ ]:
feature_coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": logistic_model.coef_[0]
})

feature_coefficients["AbsoluteCoefficient"] = (
    feature_coefficients["Coefficient"].abs()
)

top_features = feature_coefficients.sort_values(
    "AbsoluteCoefficient",
    ascending=False
).head(15)

display(top_features)


In [ ]:
plt.figure(figsize=(10, 7))
plot_data = top_features.sort_values("Coefficient")

sns.barplot(
    data=plot_data,
    x="Coefficient",
    y="Feature"
)
plt.title("Top Logistic Regression Feature Coefficients")
plt.xlabel("Coefficient")
plt.ylabel("Feature")
plt.show()


## 15. Client Risk Insights

The trained model produces a probability of churn for each customer.

For a simple business-oriented risk system:
- **Lower Risk:** probability below 0.40
- **Moderate Risk:** probability from 0.40 to below 0.70
- **Higher Risk:** probability of 0.70 or above

These thresholds are simple educational rules and should be validated with the business before being used operationally.


In [ ]:
# Generate probabilities for the complete customer dataset
X_all_scaled = scaler.transform(X)
df["ChurnProbability"] = logistic_model.predict_proba(X_all_scaled)[:, 1]

df["RiskCategory"] = pd.cut(
    df["ChurnProbability"],
    bins=[-0.01, 0.40, 0.70, 1.00],
    labels=["Lower Risk", "Moderate Risk", "Higher Risk"]
)

risk_summary = df["RiskCategory"].value_counts().reindex(
    ["Lower Risk", "Moderate Risk", "Higher Risk"]
)

display(risk_summary.to_frame("Customer Count"))


In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(
    data=df,
    x="RiskCategory",
    order=["Lower Risk", "Moderate Risk", "Higher Risk"]
)
plt.title("Customer Churn Risk Categories")
plt.xlabel("Risk Category")
plt.ylabel("Number of Customers")
plt.show()


### Practical use of risk groups

The client can use the groups to prioritise retention resources:
- **Higher Risk:** immediate review and proactive retention contact.
- **Moderate Risk:** targeted offers, support or engagement campaigns.
- **Lower Risk:** regular customer experience and loyalty activities.

The risk score should support human/business decisions rather than automatically determining customer treatment.


## 16. Business Recommendations

Based on the exploratory analysis and model workflow, the client can consider the following actions:

1. **Target customers with higher predicted churn probability.** Use the risk groups to prioritise the retention team's limited time.

2. **Focus on contract-related retention.** If shorter-contract customers show higher churn in the analysis, provide incentives for customers to move toward longer commitments.

3. **Pay attention to new customers.** Analyse customers with low tenure as an early-retention segment and provide onboarding and support during the first months.

4. **Review high monthly-charge segments.** Where the analysis shows higher churn among customers with higher charges, investigate pricing, perceived value and plan fit.

5. **Investigate service-specific churn.** If certain internet/service categories show higher churn, review service quality, customer support and plan value for those segments.

6. **Use churn probabilities for proactive retention.** Instead of treating every customer equally, use the model as a prioritisation tool for retention campaigns.

> Recommendations should be refined using the actual results produced by this notebook. Observed associations do not prove that changing one feature will necessarily cause churn to decrease.


## 17. Final Conclusion

This project implemented an end-to-end introductory AI/ML workflow for customer churn:

- loaded and explored customer data,
- cleaned missing and incorrect data types,
- analysed customer behaviour,
- created meaningful visualisations,
- calculated descriptive statistics and IQR,
- encoded and standardised features,
- split the data into training and testing sets,
- trained a Logistic Regression churn classifier,
- evaluated it using accuracy, precision, recall, F1-score and a confusion matrix,
- built a simple Linear Regression extension,
- interpreted model features,
- grouped customers into practical risk categories,
- and developed business recommendations.

The model should be treated as an educational decision-support system. Before production use, the client should validate thresholds, monitor model performance over time and test whether retention actions actually improve customer outcomes.


## 18. Submission Checklist

Before submitting, confirm that:

- [ ] Student name is filled in.
- [ ] Notebook runs from beginning to end without unexplained errors.
- [ ] Dataset/source reference is included.
- [ ] Data cleaning is explained.
- [ ] At least 5 visualisations are present.
- [ ] Statistical analysis and IQR are present.
- [ ] Preprocessing is shown.
- [ ] Train-test split is explained.
- [ ] Logistic Regression is trained.
- [ ] Accuracy is reported.
- [ ] Precision is reported.
- [ ] Recall is reported.
- [ ] F1-score is reported.
- [ ] Confusion matrix is shown.
- [ ] Linear Regression is included.
- [ ] Regression metrics are reported.
- [ ] Feature interpretation is included.
- [ ] Risk categories are included.
- [ ] At least 5 business recommendations are included.
- [ ] The notebook has been run from top to bottom.
